# 03 · Benchmark and real-world impact
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyu20sharma/fastdiff-med/blob/main/notebooks/03_benchmark_and_impact.ipynb)

Three questions:
1. **Quality:** how good are the samples at each step budget? FID against the real test set, plus the
   class accuracy of samples scored by an independent "judge" classifier.
2. **Speed:** network calls per image, batched ms per image and single-request latency on a T4.
3. **Does it matter in practice?** Train a ResNet-18 skin-lesion classifier on real data alone vs. real
   data with **rare classes topped up by 4-step student samples**, and compare balanced accuracy and
   macro-F1 (3 seeds, with 100% and 20% of the real data). A *real + oversampling* arm is included as
   the obvious cheap baseline.

All results are written to `results/` on Drive, and `docs/phase3_results.md` is generated from them.

### Setup
Works on **Google Colab** (saves to `MyDrive/fastdiff-med/`) and on **Kaggle** (saves to
`/kaggle/working/fastdiff-med/` and picks up uploaded checkpoints from `/kaggle/input/`). It clones or
updates the repo and installs dependencies. Training and distillation resume from their last
checkpoint, so after a disconnect you only rerun this notebook.

* `QUICK = True`: ~5-minute smoke test with tiny settings (outputs go to a separate `quick/` folder).
* `LITE = True`: about half the GPU time (fewer FID samples, shorter distillation rounds). The results
  are still valid, just a bit less precise.

*Colab: Runtime → Change runtime type → **T4 GPU**. Kaggle: Settings → Accelerator → **GPU T4**, Internet **on**.*

In [ ]:
import os, sys, glob, shutil
QUICK = False   # True = tiny smoke-test settings (~5 min)
LITE = True     # True = about half the GPU time (recommended if GPU time is limited)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = os.path.exists('/kaggle/working')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, CODE = '/content/drive/MyDrive/fastdiff-med', '/content/fastdiff-med'
elif IN_KAGGLE:
    BASE, CODE = '/kaggle/working/fastdiff-med', '/kaggle/working/fastdiff-med-code'
else:  # local Jupyter, started from the repo or its notebooks/ folder
    CODE = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    BASE = os.path.join(CODE, 'workdir')

if IN_COLAB or IN_KAGGLE:
    if not os.path.exists(CODE):
        !git clone -q https://github.com/shreyu20sharma/fastdiff-med.git {CODE}
    %cd {CODE}
    !git pull -q
    !pip install -q -r requirements.txt
else:
    %cd {CODE}
if CODE not in sys.path:
    sys.path.insert(0, CODE)

ROOT = BASE + ('/quick' if QUICK else '')
CKPT, RESULTS, DOCS = f'{ROOT}/checkpoints', f'{ROOT}/results', f'{ROOT}/docs'
os.environ['MEDMNIST_ROOT'] = f'{BASE}/data'  # dataset cache (~20 MB)
for d in (CKPT, RESULTS, DOCS, os.environ['MEDMNIST_ROOT']):
    os.makedirs(d, exist_ok=True)

if IN_KAGGLE:  # copy checkpoints from an uploaded Kaggle dataset, keeping the A/ B/ D/ layout
    import zipfile
    for zf in glob.glob('/kaggle/input/**/*.zip', recursive=True):  # in case Kaggle kept the zip
        zipfile.ZipFile(zf).extractall('/kaggle/working/_upload')
    for f in glob.glob('/kaggle/input/**/*', recursive=True) + glob.glob('/kaggle/working/_upload/**/*', recursive=True):
        rel = f.split('checkpoints/', 1)[-1] if 'checkpoints/' in f else None
        if rel and os.path.isfile(f) and not os.path.exists(f'{CKPT}/{rel}'):
            os.makedirs(os.path.dirname(f'{CKPT}/{rel}'), exist_ok=True)
            shutil.copy(f, f'{CKPT}/{rel}')
    print('checkpoints:', sorted(glob.glob(f'{CKPT}/**/*.pt', recursive=True)))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (no GPU: only the light cells will work)')
print('Saving to', ROOT)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
def show(path, width=None):
    if os.path.exists(path):
        display(Image(filename=path, width=width))
    else:
        print('missing:', path)

## 1. Judge classifier
A ResNet-18 trained on real DermaMNIST with class-weighted loss. It scores whether a generated
"melanoma" actually looks like a melanoma. Its own test accuracy is reported so sample accuracy can be
read relative to it.

In [ ]:
JUDGE_EPOCHS = 2 if QUICK else (25 if LITE else 40)
if not os.path.exists(f'{CKPT}/judge.pt'):
    !python -m src.judge_classifier --ckpt_dir {CKPT} --results_dir {RESULTS} --epochs {JUDGE_EPOCHS}
else:
    print('judge.pt exists, skipping')

## 2. Quality vs. speed benchmark
Every trained run is evaluated with DDIM (trailing timesteps) at 5, 10, 20 and 50 steps, the
baseline is also evaluated with the original 1000-step DDPM, and every distilled student is evaluated
at its own step count. FID uses 5,000 samples per configuration; the 1000-step baseline uses 1,000
(it is about 50× slower).

In [ ]:
if QUICK:
    !python -m src.evaluate --ckpt_dir {CKPT} --results_dir {RESULTS} --docs_dir {DOCS} --n_fid 200 --n_fid_ddpm 20 --steps 5 20
elif LITE:
    !python -m src.evaluate --ckpt_dir {CKPT} --results_dir {RESULTS} --docs_dir {DOCS} --n_fid 2000 --n_fid_ddpm 500 --steps 5 10 20
else:
    !python -m src.evaluate --ckpt_dir {CKPT} --results_dir {RESULTS} --docs_dir {DOCS} --n_fid 5000 --n_fid_ddpm 1000 --steps 5 10 20 50

In [ ]:
import pandas as pd
bench = pd.read_csv(f'{RESULTS}/benchmark.csv')
display(bench[['name', 'schedule', 'pred', 'sampler', 'steps', 'nfe', 'fid', 'class_acc', 'ms_per_image', 'single_latency_ms']].round(3))
show(f'{RESULTS}/fid_vs_steps.png', 750)
show(f'{RESULTS}/latency.png', 750)
show(f'{RESULTS}/sample_grid_compare.png', 900)

## 3. Real-world impact: synthetic augmentation of rare lesions
Every class except the largest is topped up to the size of the second-largest class (by default with
the 4-step student). A ResNet-18 is trained with 3 seeds on each arm and evaluated on the official
test split. We also time how long the synthetic set takes with the student compared with the
1000-step baseline; the baseline time is extrapolated from a subset.

In [ ]:
if QUICK:
    !python -m src.augment_experiment --ckpt_dir {CKPT} --results_dir {RESULTS} --docs_dir {DOCS} --epochs 2 --seeds 0 --baseline_time_n 8
elif LITE:
    !python -m src.augment_experiment --ckpt_dir {CKPT} --results_dir {RESULTS} --docs_dir {DOCS} --epochs 20 --seeds 0 1 2 --baseline_time_n 100
else:
    !python -m src.augment_experiment --ckpt_dir {CKPT} --results_dir {RESULTS} --docs_dir {DOCS} --epochs 30 --seeds 0 1 2 --baseline_time_n 250

In [ ]:
summ = pd.read_csv(f'{RESULTS}/augmentation_summary.csv')
display(summ[['setting', 'arm', 'n_train', 'balanced_accuracy_mean', 'balanced_accuracy_std', 'macro_f1_mean', 'macro_f1_std']].round(3))
display(pd.read_csv(f'{RESULTS}/gen_time.csv').round(1))
show(f'{RESULTS}/augmentation.png', 900)
show(f'{RESULTS}/grids/augment_synthetic.png', 400)

## 4. Results report
`docs/phase3_results.md` (on Drive) is regenerated from the CSVs. To publish, copy
`MyDrive/fastdiff-med/results/` into the repo's `results/` folder and the report into `docs/`.

In [ ]:
!python -m src.report --results_dir {RESULTS} --docs_dir {DOCS}
from IPython.display import Markdown
display(Markdown(open(f'{DOCS}/phase3_results.md', encoding='utf-8').read().split('## Benchmark')[0]))